# Tutorial: The MLflow MCP Registry

![The MLflow MCP Registry](images/mlflow_mcp_registry.svg)

## Single governed catalog of MCP servers and their tools

As teams adopt MCP, they end up with many servers — some they build, many are third-party. The **MLflow MCP
Registry** (new in MLflow 3.15) is a catalog, backed by your MLflow tracking server, for **registering**
those servers, **versioning** them, **snapshotting** their tool definitions via auto-discovery, **aliasing**
versions (e.g. `production`), and recording **access endpoints**.

It answers, in a central place: *which MCP servers
exist, what tools do they offer, and where do I reach them?*

### The three registry entities
| Entity | What it is |
|--------|------------|
| `MCPServer` | The logical server, identified by a namespaced name like `io.github.org/slug`. |
| `MCPServerVersion` | A semver'd version holding the `server.json`, a **snapshot of the server's tools**, and a status. |
| `MCPAccessEndpoint` | A concrete URL + transport clients use to reach a running instance, pinned to a version or alias. |

Each entity has its own identity and lifecycle; the values stored *on* it are its **properties** (fields you
read or set through the entity, not as separate objects):

| Entity | Its properties |
|--------|----------------|
| `MCPServer` | `name`, `description`, `display_name`, `icons`, `tags`, `aliases`, `latest_version`, `status`, timestamps |
| `MCPServerVersion` | `version`, `status`, `server_json`, `tools` (the snapshot), `source`, `connect_options` |
| `MCPAccessEndpoint` | `id`, `url`, `transport_type`, `server_version` / `server_alias` |

A version moves through a status lifecycle: **`draft → active → deprecated → deleted`**.

### What You'll Learn
- Register a **custom** MCP server you author, with live tool **auto-discovery**.
- Register an **external** (public, third-party) MCP server the same way.
- Manage the catalog: **versions**, **aliases**, **tags**, **access endpoints**, and re-discovery.
- Browse it all in the MLflow UI.

### Prerequisites
- The `mlflow[mcp]` extra installed (already in this repo's `pyproject.toml`; brings in `fastmcp`).
- **An MLflow tracking server running on the 3.15 schema** — the registry lives in the tracking store:
  ```bash
  uv run mlflow server --backend-store-uri sqlite:///mlflow.db --port 5000
  ```

> **Note:** The MCP Registry API is **experimental** in MLflow 3.15 (`@experimental(version="3.15.0")`) — the
> surface may evolve. The companion tutorial, [The MLflow MCP Server](./mlflow_mcp_server.ipynb), covers *using*
> MLflow's own MCP tools.

### Estimated Time: 20–25 minutes

---
## Step 1: Setup & verify

The registry is stored in the tracking server, so this must point at a server on the MLflow 3.15 schema.

In [ ]:
import os
import sys
import mlflow
from dotenv import load_dotenv
from packaging.version import Version
import fastmcp  # provided by the mlflow[mcp] extra

load_dotenv()

TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000")
os.environ["MLFLOW_TRACKING_URI"] = TRACKING_URI
mlflow.set_tracking_uri(TRACKING_URI)
# No experiment needed: registry entities (servers, versions, access endpoints) live in the
# tracking store itself, not inside an experiment — so unlike the MCP Server tutorial (which
# seeds traces into an experiment), there's no mlflow.set_experiment(...) call here.

assert Version(mlflow.__version__) >= Version("3.15"), (
    f"The MCP Registry needs MLflow >= 3.15 (found {mlflow.__version__})."
)

print(f"✅ MLflow version:   {mlflow.__version__}")
print(f"✅ fastmcp version:  {fastmcp.__version__}")
print(f"✅ Tracking URI:     {TRACKING_URI}")

---
## Step 2: A cleanup helper for idempotency

A server can't be deleted while it has an **active** version, and a version can't jump straight from `active`
to `deleted`. So to make this notebook safely re-runnable, we define a helper that **deprecates → deletes each
version → deletes the server**.

In [2]:
import asyncio
import nest_asyncio

from mlflow.genai import (
    register_mcp_server,
    get_mcp_server,
    get_mcp_server_version_by_alias,
    set_mcp_server_alias,
    set_mcp_server_tag,
    search_mcp_servers,
    search_mcp_server_versions,
    search_mcp_access_endpoints,
    update_mcp_server_version,
    delete_mcp_server_version,
    delete_mcp_server,
    refresh_mcp_server_version_tools,
)
nest_asyncio.apply()  # allow asyncio.run() inside Jupyter's running loop


def deregister_mcp_server(name: str) -> None:
    """Fully remove a server so the notebook is re-runnable. Every step is best-effort."""
    try:
        get_mcp_server(name)  # raises if the server doesn't exist yet
    except Exception:
        return
    for v in search_mcp_server_versions(name):
        try:
            update_mcp_server_version(name, v.version, status="deprecated")
        except Exception:
            pass
        try:
            delete_mcp_server_version(name, v.version)
        except Exception:
            pass
    try:
        delete_mcp_server(name)
    except Exception:
        pass


print("✅ Helper ready")

✅ Helper ready


---
## Step 3: Register a *custom* MCP server

The registry catalogs any MCP server via its **`server.json`** (name, version, and one or more `remotes`).
When you register with `tools` left unset, MLflow performs **live auto-discovery**: it connects to the first
`remotes[]` URL and snapshots the server's tool list into the `MCPServerVersion`.

To make discovery real, we first stand up a **custom** server — an **orders-analytics** tool, the most common
reason teams build a custom MCP server: *letting an assistant query their data*. It wraps a small in-memory
SQLite orders table and exposes `run_sql(query)` and `top_products(limit)`.

The server lives in [`utils/orders_analytics_mcp.py`](utils/orders_analytics_mcp.py) (you can run it standalone
with `uv run python utils/orders_analytics_mcp.py`); here we just launch it as a subprocess.

> Contrast it with the external server we register next: custom MCP serves **structured data**, DeepWiki serves
> **unstructured docs** — exactly the heterogeneous fleet a registry exists to catalog.

In [3]:
import subprocess
import socket
import time
from pathlib import Path

# Must match HOST / PORT in utils/orders_analytics_mcp.py.
CUSTOM_HOST, CUSTOM_PORT = "127.0.0.1", 8123
CUSTOM_URL = f"http://{CUSTOM_HOST}:{CUSTOM_PORT}/mcp"

# Launch the committed custom server as a background process
# (logs silenced so they don't clutter the notebook).
SERVER_PATH = Path("utils/orders_analytics_mcp.py")
custom_proc = subprocess.Popen(
    [sys.executable, str(SERVER_PATH)],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

#
# this function is used to wait for the custom MCP server to start
# it waits for the port to be opened on the host
def wait_for_port(host, port, timeout=15):
    """Wait for a port to be opened on a host."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        with socket.socket() as s:
            s.settimeout(1)
            if s.connect_ex((host, port)) == 0:
                return True
        time.sleep(0.3)
    return False


assert wait_for_port(CUSTOM_HOST, CUSTOM_PORT), "custom MCP server did not start"
print(f"✅ Custom MCP server listening at {CUSTOM_URL} (pid={custom_proc.pid})")

✅ Custom MCP server listening at http://127.0.0.1:8123/mcp (pid=12099)


In [4]:
# Reverse-DNS names are recommended for MCP servers.
CUSTOM_NAME = "io.github.example/orders-analytics"

# Idempotent: fully remove any prior run's registration so this cell re-runs cleanly.
deregister_mcp_server(CUSTOM_NAME)

custom_server_json = {
    "name": CUSTOM_NAME,
    "version": "1.0.0",
    "description": "Orders analytics MCP server: query an orders table via SQL.",
    "remotes": [{"url": CUSTOM_URL, "type": "streamable-http"}],
}

# tools left unset -> MLflow auto-discovers them from remotes[0] at registration time.
custom_version = register_mcp_server(
    server_json=custom_server_json,
    status="active",
    create_access_endpoints_from_remotes=True,  # also record an access endpoint per remote
)

print(f"✅ Registered '{custom_version.name}' v{custom_version.version} (status={custom_version.status})")
print(f"   Auto-discovered tools: {[t.name for t in (custom_version.tools or [])]}")

# Give the active version a friendly alias and a tag for governance.
set_mcp_server_alias(CUSTOM_NAME, "staging", custom_version.version)
set_mcp_server_tag(CUSTOM_NAME, "team", "mcp-genai-tutorials")

server = get_mcp_server(CUSTOM_NAME)
print(f"   Alias 'staging' -> v{server.aliases.get('staging')}")
print(f"   Tags: {server.tags}")

✅ Registered 'io.github.example/orders-analytics' v1.0.0 (status=active)
   Auto-discovered tools: ['run_sql', 'top_products']
   Alias 'staging' -> v1.0.0
   Tags: {'team': 'mcp-genai-tutorials'}


### From registry entry to a live tool call

Registration recorded an **access endpoint** (because we passed `create_access_endpoints_from_remotes=True`).
A consumer can look that endpoint up in the registry and connect straight to it — here we resolve it and call
the `top_products` tool we discovered, closing the loop from *register* → *discover* → *use*.

In [5]:
from fastmcp import Client
from fastmcp.client.transports import StreamableHttpTransport

# Resolve the endpoint the registry recorded for our custom server.
endpoint = search_mcp_access_endpoints(server_name=CUSTOM_NAME)[0]
print(f"🔗 Access endpoint: {endpoint.url} ({endpoint.transport_type})")


async def query_orders():
    async with Client(StreamableHttpTransport(endpoint.url)) as client:
        top = await client.call_tool("top_products", {"limit": 3})
        print("\n🏆 Top products by revenue:")
        for row in top.data:
            print(f"   {row['product']:<28} ${row['revenue']:>8,.2f}  ({row['units']} units)")


asyncio.run(query_orders())

🔗 Access endpoint: http://127.0.0.1:8123/mcp (streamable-http)

🏆 Top products by revenue:
   Aurora Standing Desk         $1,920.00  (4 units)
   Volt 27in Monitor            $1,860.00  (6 units)
   Nimbus Office Chair          $1,100.00  (5 units)


---
## Step 4: Register an *external* MCP server

Registering a third-party server is the same call — you just point `remotes[]` at *their* endpoint. Here we
register **DeepWiki**, a public MCP server (no auth) that answers questions about GitHub repositories; live
discovery snapshots its real tools.

If the server publishes a `server.json` at a URL, `register_mcp_server_from_url(url=..., status=...)` fetches
and registers it in one call.

In [6]:
# Reverse-DNS names are recommended for MCP servers.
DEEPWIKI_NAME = "com.deepwiki/mcp"

# Idempotent cleanup so this cell re-runs cleanly.
deregister_mcp_server(DEEPWIKI_NAME)

deepwiki_server_json = {
    "name": DEEPWIKI_NAME,
    "version": "1.0.0",
    "description": "DeepWiki public MCP server — ask questions about GitHub repositories.",
    "remotes": [{"url": "https://mcp.deepwiki.com/mcp", "type": "streamable-http"}],
}

try:
    deepwiki_version = register_mcp_server(
        server_json=deepwiki_server_json,
        status="active",
        create_access_endpoints_from_remotes=True,  # record an endpoint so we can call it below
    )
    print(f"✅ Registered '{deepwiki_version.name}' v{deepwiki_version.version} (status={deepwiki_version.status})")
    print(f"   Auto-discovered tools: {[t.name for t in (deepwiki_version.tools or [])]}")
except Exception as e:
    print(f"❌ Live discovery failed ({type(e).__name__}: {e}).")
    print("   See the fallback below — you can register with an explicit tool list instead.")

✅ Registered 'com.deepwiki/mcp' v1.0.0 (status=active)
   Auto-discovered tools: ['ask_question', 'read_wiki_contents', 'read_wiki_structure']


### Query the external server

Two ways to use a registered server's tools — the same two we showed for the MLflow MCP Server: call it
**programmatically**, or wire it into **Claude / your editor**.

**1. Programmatically** — resolve the endpoint the registry recorded for DeepWiki, connect to it, and call its
`ask_question` tool. DeepWiki answers questions about any public GitHub repo — here we ask about the MCP Python
SDK. (This is a live call to a remote LLM-backed service, so it takes a few seconds; it's wrapped so a network
hiccup won't break the notebook.)

In [7]:
deepwiki_endpoint = search_mcp_access_endpoints(server_name=DEEPWIKI_NAME)[0]
print(f"🔗 Access endpoint: {deepwiki_endpoint.url} ({deepwiki_endpoint.transport_type})")


async def ask_deepwiki(repo: str, question: str) -> str:
    async with Client(StreamableHttpTransport(deepwiki_endpoint.url)) as client:
        result = await client.call_tool("ask_question", {"repoName": repo, "question": question})
        return result.content[0].text if result.content else str(result.data)


try:
    answer = asyncio.run(ask_deepwiki(
        "modelcontextprotocol/python-sdk",
        "What transport types does the MCP server support?",
    ))
    print("\n❓ What transport types does the MCP server support?\n")
    print(answer[:800])
except Exception as e:
    print(f"   (DeepWiki call failed — the remote may be slow/unavailable: {type(e).__name__}: {e})")

🔗 Access endpoint: https://mcp.deepwiki.com/mcp (streamable-http)

❓ What transport types does the MCP server support?

The MCP server supports three transport types: `stdio`, `streamable-http`, and `sse` (Server-Sent Events).   While `sse` is still supported for legacy clients, `streamable-http` is the recommended HTTP transport. 

### Supported Transport Types

*   **`stdio`**: This transport is used for local servers where the host launches the server as a subprocess and communicates over its standard input and output.   It is the default transport when no other is specified. 
*   **`streamable-http`**: This transport provides a full HTTP server listening on a port, suitable for deployed applications.  It was introduced as a replacement for `sse` in the 2025-03-26 protocol revision. 
*   **`sse` (Server-Sent Events)**: This is an older HTTP transport that is still supported for compatibility with older c


**2. From Claude / your editor** — you usually don't hand-write a client; you register the server once and
its tools become available to your assistant. DeepWiki is a **remote HTTP** MCP server, so (unlike the MLflow
stdio server) you register it **by URL** — no subprocess, no `command`.

**Claude Code / Claude Desktop:**
```bash
claude mcp add --transport http deepwiki https://mcp.deepwiki.com/mcp
```

**Claude — project-level `.mcp.json`:**
```json
{
  "mcpServers": {
    "deepwiki": { "type": "http", "url": "https://mcp.deepwiki.com/mcp" }
  }
}
```

**VS Code — `.vscode/mcp.json`** (note the `servers` key):
```json
{
  "servers": {
    "deepwiki": { "type": "http", "url": "https://mcp.deepwiki.com/mcp" }
  }
}
```

Then just ask in natural language, and the assistant calls `ask_question` for you:
- *"Using DeepWiki, what transport types does the modelcontextprotocol/python-sdk server support?"*
- *"Ask DeepWiki how authentication works in the openai/openai-python repo."*

---
## Step 5: Versioning, aliases & management

Registries earn their keep over time. Register a **new version** of the custom server, then re-point the
`production` alias to it — consumers that resolve `production` follow along without changing their config.

In [8]:
# Register v1.1.0 of the same server (same remote → same tools re-discovered).
v11 = register_mcp_server(
    server_json={**custom_server_json, "version": "1.1.0"},
    status="active",
)
print(f"✅ Registered {v11.name} v{v11.version}")

# Promote the new version: point the 'production' alias at 1.1.0.
set_mcp_server_alias(CUSTOM_NAME, "production", "1.1.0")
print(f"⭐ Alias 'production' now -> v{get_mcp_server_version_by_alias(CUSTOM_NAME, 'production').version}")

✅ Registered io.github.example/orders-analytics v1.1.0
⭐ Alias 'production' now -> v1.1.0


In [9]:
print("📚 Registered MCP servers:")
for s in search_mcp_servers():
    print(f"   - {s.name} (status={s.status}, latest=v{s.latest_version})")

print(f"\n🔖 Versions of {CUSTOM_NAME}:")
for v in search_mcp_server_versions(CUSTOM_NAME):
    print(f"   - v{v.version}  status={v.status}  tools={[t.name for t in (v.tools or [])]}")

print(f"\n🔗 Access endpoints for {CUSTOM_NAME}:")
for ep in search_mcp_access_endpoints(server_name=CUSTOM_NAME):
    print(f"   - {ep.url} ({ep.transport_type})")

# Re-discover tools for a version (dry-run shows what *would* be snapshotted, without saving).
refreshed = refresh_mcp_server_version_tools(CUSTOM_NAME, "1.1.0", dry_run=True)
print(f"\n🔄 refresh v1.1.0 (dry_run) would snapshot: {[t.name for t in (refreshed.tools or [])]}")

📚 Registered MCP servers:
   - com.deepwiki/mcp (status=active, latest=v1.0.0)
   - io.github.example/orders-analytics (status=active, latest=v1.1.0)

🔖 Versions of io.github.example/orders-analytics:
   - v1.0.0  status=active  tools=['run_sql', 'top_products']
   - v1.1.0  status=active  tools=['run_sql', 'top_products']

🔗 Access endpoints for io.github.example/orders-analytics:
   - http://127.0.0.1:8123/mcp (streamable-http)

🔄 refresh v1.1.0 (dry_run) would snapshot: ['run_sql', 'top_products']


### In the MLflow UI

Open the tracking server in a browser (**http://localhost:5000**) and select the **MCP Registry** section in
the sidebar. You'll see both registered servers; drill into a version to inspect its `server.json` and tool
snapshot, and view its aliases and access endpoints.

---
## Step 6: Cleanup (optional)

Remove the demo registrations and stop the local custom server.

In [10]:
for name in [CUSTOM_NAME, DEEPWIKI_NAME]:
    deregister_mcp_server(name)  # deprecate -> delete versions -> delete server
    print(f"🗑️  Removed registry entry: {name}")

# Stop the local custom MCP server process.
try:
    custom_proc.terminate()
    custom_proc.wait(timeout=5)
    print(f"🛑 Stopped custom MCP server (pid={custom_proc.pid})")
except Exception as e:
    print(f"   (custom server already stopped: {e})")

🗑️  Removed registry entry: io.github.example/orders-analytics
🗑️  Removed registry entry: com.deepwiki/mcp
🛑 Stopped custom MCP server (pid=12099)


---
## Summary

You used the **MLflow MCP Registry** as a governed catalog:

- **Registered a custom MCP server** (orders-analytics) with live tool auto-discovery, an alias, and a tag —
  then resolved its access endpoint and called a discovered tool (register → discover → use).
- **Registered an external MCP server** (DeepWiki) the same way, then **queried it** through its endpoint with
  `ask_question` — with a fallback for offline / authenticated remotes.
- **Managed versions and aliases** — added v1.1.0, re-pointed `production`, listed servers/versions/endpoints,
  and did a dry-run tool refresh — and viewed it all in the MLflow UI.

Together with **[The MLflow MCP Server](./mlflow_mcp_server.ipynb)**, you now have both halves of MLflow's MCP
story: turning MLflow into tools, and cataloging the tools your team relies on.

### Next Steps
- [The MLflow MCP Server](./mlflow_mcp_server.ipynb) — use MLflow's own built-in MCP tools.
- [MLflow docs](https://mlflow.org/docs/latest/): **MCP Registry** (`/genai/mcp-registry/`) and **MCP Server** (`/genai/mcp/`).